In [1]:
import pandas as pd
import numpy as np

PROFILES_PATH = "../data/raw/pandora_profiles.csv"
COMMENTS_PATH = "../data/raw/pandora_comments.csv"
OUTPUT_PATH   = "../data/processed/"

BIG5 = ["agreeableness", "openness", "conscientiousness", "extraversion", "neuroticism"]

In [2]:
profiles = pd.read_csv(PROFILES_PATH, low_memory=False)

# Keep only users where ALL 5 traits are present
profiles_clean = profiles[BIG5 + ["author"]].dropna(subset=BIG5).copy()

print(f"Total users in profiles:         {len(profiles)}")
print(f"Users with complete Big Five:    {len(profiles_clean)}")

Total users in profiles:         10295
Users with complete Big Five:    1568


In [3]:
# >= 50 = High (1), < 50 = Low (0)
# This is what the PANDORA baseline paper does too
for trait in BIG5:
    profiles_clean[f"{trait}_label"] = (profiles_clean[trait] >= 50).astype(int)

label_cols = [f"{t}_label" for t in BIG5]

# Sanity check: class distribution per trait
print("Label distribution (1 = High):")
print(profiles_clean[label_cols].mean().round(3))

Label distribution (1 = High):
agreeableness_label        0.435
openness_label             0.710
conscientiousness_label    0.378
extraversion_label         0.350
neuroticism_label          0.504
dtype: float64


In [4]:
# This file is 17M rows — this will take a minute
print("Loading comments... (this takes a while)")
comments = pd.read_csv(COMMENTS_PATH, low_memory=False)
print(f"Total comments: {len(comments)}")

# English only
comments_en = comments[comments["lang"] == "en"].copy()
print(f"English comments: {len(comments_en)}")

# Keep only comments from users that have Big Five labels
valid_authors = set(profiles_clean["author"])
comments_filtered = comments_en[comments_en["author"].isin(valid_authors)].copy()
print(f"Comments from labeled users: {len(comments_filtered)}")


Loading comments... (this takes a while)
Total comments: 17640062
English comments: 16637211
Comments from labeled users: 2830311


In [5]:
# Sort by time so we take the most recent 50 comments per user
comments_filtered = comments_filtered.sort_values("created_utc", ascending=False)
comments_capped = (
    comments_filtered
    .groupby("author", group_keys=False)
    .head(50)
)

print(f"Comments after capping at 50/user: {len(comments_capped)}")
print(f"Unique users with comments: {comments_capped['author'].nunique()}")

Comments after capping at 50/user: 67450
Unique users with comments: 1568


In [6]:
users_with_comments = set(comments_capped["author"])
users_with_labels   = set(profiles_clean["author"])
overlap = users_with_labels & users_with_comments

print(f"Users with Big Five labels:         {len(users_with_labels)}")
print(f"Users also found in comments:       {len(overlap)}")
print(f"Users with labels but NO comments:  {len(users_with_labels - users_with_comments)}")

Users with Big Five labels:         1568
Users also found in comments:       1568
Users with labels but NO comments:  0


In [7]:
# Keep only users present in both
profiles_final = profiles_clean[profiles_clean["author"].isin(overlap)].copy()

# Save processed files
profiles_final[["author"] + BIG5 + label_cols].to_csv(
    OUTPUT_PATH + "profiles_labeled.csv", index=False
)
comments_capped[["author", "body", "created_utc"]].to_csv(
    OUTPUT_PATH + "comments_capped.csv", index=False
)

print("Saved:")
print(f"  profiles_labeled.csv  — {len(profiles_final)} users")
print(f"  comments_capped.csv   — {len(comments_capped)} comments")

Saved:
  profiles_labeled.csv  — 1568 users
  comments_capped.csv   — 67450 comments


In [11]:
from skmultilearn.model_selection import iterative_train_test_split
import numpy as np
import json

# Force authors to a plain numpy object array — skmultilearn needs this
authors = np.array(profiles_final["author"].values, dtype=object).reshape(-1, 1)
labels  = profiles_final[label_cols].values.astype(int)  # (N, 5) numpy int array

# Step 1: 70% train, 30% temp
authors_train, labels_train, authors_temp, labels_temp = iterative_train_test_split(
    authors, labels, test_size=0.30
)

# Step 2: split temp 50/50 → val and test
authors_val, labels_val, authors_test, labels_test = iterative_train_test_split(
    authors_temp, labels_temp, test_size=0.50
)

# These are now plain numpy arrays — flatten works fine
train_users = authors_train.flatten().tolist()
val_users   = authors_val.flatten().tolist()
test_users  = authors_test.flatten().tolist()

print(f"Train: {len(train_users)} users")
print(f"Val:   {len(val_users)} users")
print(f"Test:  {len(test_users)} users")

Train: 1092 users
Val:   243 users
Test:  233 users


In [12]:
trait_names = ["agreeableness", "openness", "conscientiousness", "extraversion", "neuroticism"]

def show_dist(label_array, name):
    means = label_array.mean(axis=0).round(3)
    print(f"\n{name}:")
    for t, m in zip(trait_names, means):
        print(f"  {t:<20} {m:.3f}")

show_dist(labels_train, "Train")
show_dist(labels_val,   "Val")
show_dist(labels_test,  "Test")


Train:
  agreeableness        0.437
  openness             0.713
  conscientiousness    0.380
  extraversion         0.352
  neuroticism          0.507

Val:
  agreeableness        0.424
  openness             0.687
  conscientiousness    0.366
  extraversion         0.342
  neuroticism          0.486

Test:
  agreeableness        0.438
  openness             0.717
  conscientiousness    0.382
  extraversion         0.352
  neuroticism          0.511


In [13]:
# pos_weight = (# negative samples) / (# positive samples) per trait
# This is what PyTorch BCEWithLogitsLoss expects
neg = (labels_train == 0).sum(axis=0)
pos = (labels_train == 1).sum(axis=0)
pos_weights = (neg / pos).tolist()

print("pos_weight per trait (pass this to BCEWithLogitsLoss):")
for t, w in zip(trait_names, pos_weights):
    print(f"  {t:<20} {w:.3f}")

pos_weight per trait (pass this to BCEWithLogitsLoss):
  agreeableness        1.289
  openness             0.402
  conscientiousness    1.631
  extraversion         1.844
  neuroticism          0.971


In [15]:
splits = {
    "train": train_users,
    "val":   val_users,
    "test":  test_users
}

with open(OUTPUT_PATH + "splits.json", "w") as f:
    json.dump(splits, f)

with open(OUTPUT_PATH + "pos_weights.json", "w") as f:
    json.dump({"pos_weights": pos_weights, "traits": trait_names}, f)

print("Saved splits.json")
print("Saved pos_weights.json")
print(f"\nTotal users: {len(train_users) + len(val_users) + len(test_users)}")

Saved splits.json
Saved pos_weights.json

Total users: 1568


In [16]:
summary = {
    "dataset": "PANDORA (Gjurkovic et al., 2020)",
    "total_users_in_profiles": 10295,
    "users_with_complete_big5": 1568,
    "labeling_method": "Binary threshold >= 50 percentile = High (1), < 50 = Low (0)",
    "comment_filter": "English only, capped at 50 most recent per user",
    "total_comments": 67450,
    "avg_comments_per_user": round(67450 / 1568, 1),
    "split_method": "Iterative multi-label stratification (Tan et al., 2025)",
    "split_ratio": "70 / 15 / 15",
    "train_users": 1092,
    "val_users": 243,
    "test_users": 233,
    "label_distribution_train": dict(zip(trait_names, labels_train.mean(axis=0).round(3).tolist())),
    "pos_weights": dict(zip(trait_names, [round(w, 3) for w in pos_weights])),
    "imbalance_method": "pos_weight in BCEWithLogitsLoss (Tan et al., 2025)"
}

with open(OUTPUT_PATH + "pipeline_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))

{
  "dataset": "PANDORA (Gjurkovic et al., 2020)",
  "total_users_in_profiles": 10295,
  "users_with_complete_big5": 1568,
  "labeling_method": "Binary threshold >= 50 percentile = High (1), < 50 = Low (0)",
  "comment_filter": "English only, capped at 50 most recent per user",
  "total_comments": 67450,
  "avg_comments_per_user": 43.0,
  "split_method": "Iterative multi-label stratification (Tan et al., 2025)",
  "split_ratio": "70 / 15 / 15",
  "train_users": 1092,
  "val_users": 243,
  "test_users": 233,
  "label_distribution_train": {
    "agreeableness": 0.437,
    "openness": 0.713,
    "conscientiousness": 0.38,
    "extraversion": 0.352,
    "neuroticism": 0.507
  },
  "pos_weights": {
    "agreeableness": 1.289,
    "openness": 0.402,
    "conscientiousness": 1.631,
    "extraversion": 1.844,
    "neuroticism": 0.971
  },
  "imbalance_method": "pos_weight in BCEWithLogitsLoss (Tan et al., 2025)"
}
